# 01 — Exploratory Data Analysis

**Student Performance Prediction System**

This notebook covers Module 2 of the brief: understand the dataset before
modelling anything. We look at attendance, participation, resource use and
parental involvement, and check how each relates to the Low / Medium / High
performance band.

Everything here calls into `src/` — the same code the dashboard and the API
use — so nothing in this notebook can drift away from what actually ships.

In [ ]:
# Make `src` importable no matter where Jupyter was launched from.
import sys, warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

print(f"Project root: {ROOT}")

## 1. Load and clean

The raw file goes through schema validation and then cleaning: duplicate removal, whitespace trimming, missing-value imputation and range clipping.

In [ ]:
from src.data.load_data import load_raw_data, validate_schema
from src.data.preprocess import clean_dataframe
from src.utils.config import load_config

cfg = load_config()
raw = load_raw_data(cfg=cfg)
schema_report = validate_schema(raw, cfg, strict=True)

print(f"Raw shape:            {raw.shape}")
print(f"Missing values:       {schema_report['missing_values_total']}")
print(f"Exact duplicate rows: {schema_report['exact_duplicate_rows']}")
raw.head()

In [ ]:
df, cleaning_report = clean_dataframe(raw, cfg)

print(f"Rows in:  {cleaning_report['rows_in']}")
print(f"Rows out: {cleaning_report['rows_out']}")
print(f"Duplicates removed: {cleaning_report['exact_duplicates_removed']}")
print(f"Class distribution: {cleaning_report['class_distribution']}")
df.head()

## 2. What is in the dataset?

16 features and one target. The features split into three kinds:

- **Behavioural counters** (0-100): hands raised, resources opened, announcements read, discussion posts. These are the dataset's proxies for participation and study effort.
- **Attendance**: `StudentAbsenceDays`, a two-level flag for under vs. 7-or-more days absent.
- **Context and demographics**: nationality, subject, grade, section, and which parent is responsible — plus two parental-engagement survey fields.

In [ ]:
overview = df.describe(include="all").T
overview["dtype"] = df.dtypes.astype(str)
overview[["dtype", "count", "unique", "top", "freq", "mean", "std", "min", "max"]]

## 3. Class balance

The target is already L/M/H in the source data — no post-hoc bucketing of a continuous grade required. This is the main reason we chose this dataset over the UCI alternative.

In [ ]:
from src.analysis.eda import apply_house_style, plot_class_distribution
apply_house_style()

path, caption = plot_class_distribution(df, cfg)
print(caption)
from IPython.display import Image, display
display(Image(str(path)))

## 4. Engagement behaviour across the three bands

The question: do students in different performance bands actually *behave*
differently, or does it just feel that way?

In [ ]:
from src.analysis.eda import plot_numeric_distributions, plot_boxplots_by_class

for fn in (plot_numeric_distributions, plot_boxplots_by_class):
    p, cap = fn(df, cfg)
    display(Image(str(p)))
    print(cap, "\n")

In [ ]:
# The same comparison as a table — group means for every engagement counter.
numeric = cfg["data"]["numeric_features"]
df.groupby("Class")[numeric].agg(["mean", "median", "std"]).round(1).reindex(["L", "M", "H"])

## 5. Attendance — the headline factor

This single two-level column turns out to carry more signal than any other feature in the dataset.

In [ ]:
from src.analysis.eda import plot_absence_vs_class

p, cap = plot_absence_vs_class(df, cfg)
display(Image(str(p)))
print(cap)

In [ ]:
import pandas as pd
pd.crosstab(df["StudentAbsenceDays"], df["Class"], normalize="index").round(3)[["L", "M", "H"]]

## 6. Parental involvement and demographics

Note the gender panel below. It differs noticeably between bands — which is exactly why this project runs a formal fairness audit later rather than assuming the model is even-handed.

In [ ]:
from src.analysis.eda import plot_categorical_panel

p, cap = plot_categorical_panel(df, cfg)
display(Image(str(p)))
print(cap)

## 7. Do the engagement measures overlap?

If two features carried identical information we would want to drop one. Spearman correlation (rank-based, so it is not distorted by skew) says they are related but distinct.

In [ ]:
from src.analysis.eda import plot_correlation_heatmap, plot_engagement_scatter

for fn in (plot_correlation_heatmap, plot_engagement_scatter):
    p, cap = fn(df, cfg)
    display(Image(str(p)))
    print(cap, "\n")

## 8. Subject-wise view

Required by Module 4 (subject-wise analysis) and surfaced on the dashboard's Overview page.

In [ ]:
from src.analysis.eda import plot_topic_breakdown

p, cap = plot_topic_breakdown(df, cfg)
display(Image(str(p)))
print(cap)

## 9. What we take into modelling

1. **Attendance** is the strongest single discriminator by a wide margin.
2. **Resource use and hands raised** separate the bands almost as cleanly.
3. **Discussion posts** are far weaker than the other three counters — worth keeping, but not worth building an intervention around.
4. **Parental engagement** tracks strongly with outcomes.
5. **Gender differs across bands**, so fairness needs formal testing rather than assumption.

All five of these are eyeball impressions at this point. Notebook `02`
tests them properly.

In [ ]:
from src.analysis.eda import dataset_overview
import json
print(json.dumps(dataset_overview(df, cfg), indent=2))